# 구조화된 엔티티 장기 메모리

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `ConversationEntityMemory`는 레거시 메모리 추상화에 속합니다. 현재는
**구조화 출력으로 엔티티를 추출**하고, 애플리케이션이 정한 스키마와 정책에 따라
**LangGraph Store**에 저장하는 방식이 더 명시적이고 확장 가능합니다.

이 예제는 사람의 이름·직무·관계·계획을 JSON 구조로 추출하여 사용자별 namespace에
저장합니다. `InMemoryStore`는 실습용이며 운영 환경에서는 DB 기반 store를 사용합니다.

참고: [LangChain 장기 메모리](https://docs.langchain.com/oss/python/langchain/long-term-memory),
[LangGraph Stores](https://docs.langchain.com/oss/python/langgraph/stores)


In [1]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [2]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.4-mini")
model = init_chat_model(MODEL_ID)


## 엔티티 스키마 정의 및 추출

스키마가 곧 메모리 계약입니다. 자유 형식 요약보다 검증·병합·검색이 쉽습니다.


In [3]:
from pydantic import BaseModel, Field


class Person(BaseModel):
    name: str = Field(description="대화에 등장한 사람의 이름")
    occupation: str | None = Field(default=None, description="직무 또는 역할")
    relationship: str | None = Field(default=None, description="다른 인물과의 관계")
    plans: list[str] = Field(default_factory=list, description="명시적으로 언급된 계획")


class People(BaseModel):
    people: list[Person]


entity_extractor = model.with_structured_output(People)

conversation = (
    "테디와 셜리는 한 회사에서 일하는 동료입니다. "
    "테디는 개발자이고 셜리는 디자이너입니다. "
    "두 사람은 회사를 그만두고 함께 창업할 계획입니다."
)

extracted = entity_extractor.invoke(
    [
        {
            "role": "system",
            "content": (
                "주어진 문장에서 명시된 사람 정보만 추출하세요. "
                "추측하지 말고, 모르는 필드는 비워 두세요."
            ),
        },
        {"role": "user", "content": conversation},
    ]
)
extracted


People(people=[Person(name='테디', occupation='개발자', relationship='셜리와 같은 회사의 동료', plans=['회사를 그만두고 함께 창업']), Person(name='셜리', occupation='디자이너', relationship='테디와 같은 회사의 동료', plans=['회사를 그만두고 함께 창업'])])

## 사용자별 namespace에 upsert

namespace는 폴더처럼 메모리를 구분합니다. 여기서는 `(users, user_id, entities)`를
사용하고 사람 이름을 key로 사용합니다.


In [4]:
from langgraph.store.memory import InMemoryStore

entity_store = InMemoryStore()
user_id = "user-123"
namespace = ("users", user_id, "entities")


def upsert_person(person: Person) -> None:
    new_value = person.model_dump(exclude_none=True)
    current = entity_store.get(namespace, person.name)
    merged = {**current.value, **new_value} if current else new_value
    entity_store.put(namespace, person.name, merged)


for person in extracted.people:
    upsert_person(person)


In [5]:
# namespace에 저장된 모든 엔티티
[item.value for item in entity_store.search(namespace)]


[{'name': '테디',
  'occupation': '개발자',
  'relationship': '셜리와 같은 회사의 동료',
  'plans': ['회사를 그만두고 함께 창업']},
 {'name': '셜리',
  'occupation': '디자이너',
  'relationship': '테디와 같은 회사의 동료',
  'plans': ['회사를 그만두고 함께 창업']}]

In [6]:
# key를 알고 있을 때는 직접 조회합니다.
shirley = entity_store.get(namespace, "셜리")
shirley.value if shirley else "'셜리'라는 key가 없습니다. 추출 결과의 실제 이름을 확인하세요."


{'name': '셜리',
 'occupation': '디자이너',
 'relationship': '테디와 같은 회사의 동료',
 'plans': ['회사를 그만두고 함께 창업']}

실제 서비스에서는 다음 정책도 명시해야 합니다.

- 새 사실과 기존 사실이 충돌할 때 덮어쓸지, 이력을 남길지
- 민감 정보의 저장 허용 범위와 삭제 정책
- 사용자별 namespace 격리와 접근 제어
- 대화 중 즉시 저장할지(hot path), 백그라운드에서 정리할지(background)
